# Parte 2 — Reasoning y Function Calling en Azure AI Foundry

Este notebook cubre:
- **2.1** Modelos razonadores con distintos niveles de razonamiento (low, medium, high)
- **2.2** Function Calling con búsqueda web + función custom (⭐)

## Configuración y Credenciales

Archivo `.env` necesario:
```
AZURE_ENDPOINT=https://<tu-proyecto>.openai.azure.com/
AZURE_API_KEY=<tu-api-key>
AZURE_REASONING_DEPLOYMENT=<deployment-modelo-razonador>  # ej: o1, o3-mini
AZURE_DEPLOYMENT_NAME=<deployment-modelo-standard>         # ej: gpt-4o
AZURE_API_VERSION=2024-12-01-preview
BING_SEARCH_KEY=<tu-bing-search-api-key>                   # Para búsqueda web
```

In [17]:
import sys
!{sys.executable} -m pip install openai python-dotenv requests --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [18]:
import os
import json
import time
import requests
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_ENDPOINT"),
    api_key=os.getenv("AZURE_API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION", "2024-12-01-preview")
)

REASONING_DEPLOYMENT = os.getenv("AZURE_REASONING_DEPLOYMENT", "o3-mini")
STANDARD_DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_NAME", "gpt-4o")
BING_KEY = os.getenv("BING_SEARCH_KEY")

print(f"Reasoning model: {REASONING_DEPLOYMENT}")
print(f"Standard model: {STANDARD_DEPLOYMENT}")
print(f"Bing Search: {'configurado' if BING_KEY else 'NO configurado'}")

Reasoning model: gpt-5.4-mini
Standard model: gpt-4o
Bing Search: NO configurado


---
## 2.1 — Razonamiento con distintos niveles

Los modelos razonadores (o1, o3-mini) permiten configurar el esfuerzo de razonamiento mediante el parámetro `reasoning_effort`: `low`, `medium`, `high`.

- **low**: respuesta más rápida, menos tokens de razonamiento interno
- **medium**: equilibrio entre velocidad y profundidad
- **high**: máximo razonamiento, más lento y costoso pero más preciso en problemas complejos

In [19]:
def razonar(prompt: str, nivel: str) -> dict:
    """Llama al modelo razonador con el nivel especificado y mide tiempo y tokens."""
    print(f"\n>>> Nivel: {nivel.upper()} — ejecutando...")
    start = time.time()
    
    response = client.chat.completions.create(
        model=REASONING_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort=nivel,
        max_completion_tokens=2000
    )
    
    elapsed = time.time() - start
    content = response.choices[0].message.content
    usage = response.usage
    
    resultado = {
        "nivel": nivel,
        "respuesta": content,
        "tiempo_segundos": round(elapsed, 2),
        "tokens_totales": usage.total_tokens,
        "tokens_razonamiento": getattr(getattr(usage, 'completion_tokens_details', None), 'reasoning_tokens', 'N/A')
    }
    
    print(f"   Tiempo: {resultado['tiempo_segundos']}s | Tokens totales: {resultado['tokens_totales']}")
    return resultado


# Tarea compleja para comparar niveles
tarea = """Tengo una tabla de hechos en Azure SQL con 500 millones de registros. 
Las queries de agregación tardan más de 10 minutos. 
Propón una estrategia completa de optimización considerando: índices, particionado, 
columnstore indexes, y posible migración a Fabric o Synapse.
Razona paso a paso y prioriza las acciones por impacto."""

resultados = {}
for nivel in ["low", "medium", "high"]:
    resultados[nivel] = razonar(tarea, nivel)
    time.sleep(2)


>>> Nivel: LOW — ejecutando...
   Tiempo: 13.75s | Tokens totales: 2079

>>> Nivel: MEDIUM — ejecutando...
   Tiempo: 14.68s | Tokens totales: 2079

>>> Nivel: HIGH — ejecutando...
   Tiempo: 13.65s | Tokens totales: 2079


In [20]:
# Comparativa de resultados
print("\n" + "="*60)
print("COMPARATIVA DE NIVELES DE RAZONAMIENTO")
print("="*60)

for nivel, datos in resultados.items():
    print(f"\n### NIVEL: {nivel.upper()}")
    print(f"Tiempo: {datos['tiempo_segundos']}s")
    print(f"Tokens totales: {datos['tokens_totales']}")
    print(f"Tokens de razonamiento: {datos['tokens_razonamiento']}")
    print(f"Respuesta (primeros 500 chars):")
    print(datos['respuesta'][:500] + "..." if len(datos['respuesta']) > 500 else datos['respuesta'])

print("\n" + "="*60)
print("RESUMEN COMPARATIVO")
print("-"*40)
print(f"{'Nivel':<10} {'Tiempo':<12} {'Tokens':<12}")
print("-"*40)
for nivel, datos in resultados.items():
    print(f"{nivel:<10} {datos['tiempo_segundos']:<12} {datos['tokens_totales']:<12}")


COMPARATIVA DE NIVELES DE RAZONAMIENTO

### NIVEL: LOW
Tiempo: 13.75s
Tokens totales: 2079
Tokens de razonamiento: 2000
Respuesta (primeros 500 chars):


### NIVEL: MEDIUM
Tiempo: 14.68s
Tokens totales: 2079
Tokens de razonamiento: 2000
Respuesta (primeros 500 chars):


### NIVEL: HIGH
Tiempo: 13.65s
Tokens totales: 2079
Tokens de razonamiento: 2000
Respuesta (primeros 500 chars):


RESUMEN COMPARATIVO
----------------------------------------
Nivel      Tiempo       Tokens      
----------------------------------------
low        13.75        2079        
medium     14.68        2079        
high       13.65        2079        


---
## 2.2 — Function Calling

### Arquitectura
1. El modelo recibe el prompt y decide qué funciones llamar
2. Devuelve un `tool_call` con los argumentos
3. Ejecutamos la función real (búsqueda web o función custom)
4. Devolvemos el resultado al modelo para generar la respuesta final

Implementamos:
- `buscar_web`: búsqueda con Bing Search API
- `analizar_pipeline_datos`: función custom que analiza parámetros de un pipeline (⭐)

In [21]:
# Definición de herramientas disponibles para el modelo
herramientas = [
    {
        "type": "function",
        "function": {
            "name": "buscar_web",
            "description": "Busca información actualizada en internet usando Bing Search. Úsala para preguntas sobre eventos recientes, precios, documentación o cualquier información que pueda haber cambiado.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "La consulta de búsqueda en lenguaje natural"
                    },
                    "num_resultados": {
                        "type": "integer",
                        "description": "Número de resultados a devolver (1-5)",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analizar_pipeline_datos",
            "description": "Analiza los parámetros de un pipeline de datos y devuelve métricas estimadas de rendimiento, coste y recomendaciones de optimización.",
            "parameters": {
                "type": "object",
                "properties": {
                    "volumen_gb": {
                        "type": "number",
                        "description": "Volumen de datos en GB"
                    },
                    "frecuencia": {
                        "type": "string",
                        "description": "Frecuencia de ejecución: hourly, daily, weekly"
                    },
                    "tecnologia": {
                        "type": "string",
                        "description": "Tecnología del pipeline: spark, adf, fabric, synapse"
                    }
                },
                "required": ["volumen_gb", "frecuencia", "tecnologia"]
            }
        }
    }
]

print(f"Herramientas definidas: {len(herramientas)}")
for h in herramientas:
    print(f"  - {h['function']['name']}: {h['function']['description'][:60]}...")

Herramientas definidas: 2
  - buscar_web: Busca información actualizada en internet usando Bing Search...
  - analizar_pipeline_datos: Analiza los parámetros de un pipeline de datos y devuelve mé...


In [22]:
# Implementación de las funciones reales

def buscar_web(query: str, num_resultados: int = 3) -> str:
    """Búsqueda real con Bing Search API."""
    if not BING_KEY:
        # Mock para desarrollo sin API key
        return json.dumps({
            "nota": "Modo demo - configura BING_SEARCH_KEY para búsqueda real",
            "query": query,
            "resultados": [
                {"titulo": f"Resultado demo para: {query}", "snippet": "Contenido de ejemplo", "url": "https://example.com"}
            ]
        })
    
    url = "https://api.bing.microsoft.com/v7.0/search"
    headers = {"Ocp-Apim-Subscription-Key": BING_KEY}
    params = {"q": query, "count": num_resultados, "mkt": "es-ES"}
    
    response = requests.get(url, headers=headers, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    
    resultados = []
    for item in data.get("webPages", {}).get("value", [])[:num_resultados]:
        resultados.append({
            "titulo": item.get("name"),
            "snippet": item.get("snippet"),
            "url": item.get("url")
        })
    
    return json.dumps({"query": query, "resultados": resultados})


def analizar_pipeline_datos(volumen_gb: float, frecuencia: str, tecnologia: str) -> str:
    """Función custom que estima métricas de un pipeline de datos."""
    # Lógica de estimación basada en parámetros
    coste_por_gb = {"spark": 0.08, "adf": 0.12, "fabric": 0.06, "synapse": 0.10}
    tiempo_por_gb = {"spark": 0.5, "adf": 1.2, "fabric": 0.4, "synapse": 0.7}  # minutos por GB
    
    multiplicador_freq = {"hourly": 24*30, "daily": 30, "weekly": 4}
    
    coste_unit = coste_por_gb.get(tecnologia.lower(), 0.10)
    tiempo_unit = tiempo_por_gb.get(tecnologia.lower(), 1.0)
    freq_mult = multiplicador_freq.get(frecuencia.lower(), 30)
    
    coste_mensual = round(volumen_gb * coste_unit * freq_mult, 2)
    tiempo_ejecucion = round(volumen_gb * tiempo_unit, 1)
    
    recomendaciones = []
    if volumen_gb > 100:
        recomendaciones.append("Considera particionado de datos para mejorar rendimiento")
    if frecuencia == "hourly" and volumen_gb > 50:
        recomendaciones.append("Evalúa procesamiento incremental en lugar de full load")
    if tecnologia == "adf" and volumen_gb > 200:
        recomendaciones.append("Migración a Fabric o Spark podría reducir costes un 40%")
    
    return json.dumps({
        "analisis": {
            "volumen_gb": volumen_gb,
            "frecuencia": frecuencia,
            "tecnologia": tecnologia,
            "coste_estimado_mensual_usd": coste_mensual,
            "tiempo_ejecucion_minutos": tiempo_ejecucion,
            "ejecuciones_mensuales": freq_mult,
            "recomendaciones": recomendaciones if recomendaciones else ["Pipeline bien dimensionado"]
        }
    })


# Dispatcher de funciones
FUNCIONES_DISPONIBLES = {
    "buscar_web": buscar_web,
    "analizar_pipeline_datos": analizar_pipeline_datos
}

print("Funciones implementadas y listas.")

Funciones implementadas y listas.


In [23]:
def ejecutar_con_function_calling(prompt: str, verbose: bool = True) -> str:
    """Ejecuta un prompt con function calling completo (ciclo de herramientas)."""
    mensajes = [
        {"role": "system", "content": "Eres un experto en ingeniería de datos y Azure. Usa las herramientas disponibles cuando sea necesario para dar respuestas precisas y actualizadas."},
        {"role": "user", "content": prompt}
    ]
    
    if verbose:
        print(f"\n>>> PROMPT: {prompt}")
    
    # Primera llamada — el modelo decide si usar herramientas
    response = client.chat.completions.create(
        model=STANDARD_DEPLOYMENT,
        messages=mensajes,
        tools=herramientas,
        tool_choice="auto",
        max_tokens=1000
    )
    
    message = response.choices[0].message
    
    # Si el modelo no necesita herramientas, responde directamente
    if not message.tool_calls:
        if verbose:
            print("El modelo respondió directamente (sin herramientas)")
        return message.content
    
    # Procesar tool_calls
    mensajes.append(message)
    
    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        
        if verbose:
            print(f"\n🔧 Función llamada: {fn_name}")
            print(f"   Argumentos: {fn_args}")
        
        # Ejecutar la función real
        fn_resultado = FUNCIONES_DISPONIBLES[fn_name](**fn_args)
        
        if verbose:
            resultado_parsed = json.loads(fn_resultado)
            print(f"   Resultado: {str(resultado_parsed)[:200]}...")
        
        # Añadir resultado al historial
        mensajes.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": fn_resultado
        })
    
    # Segunda llamada — respuesta final con los resultados de las herramientas
    respuesta_final = client.chat.completions.create(
        model=STANDARD_DEPLOYMENT,
        messages=mensajes,
        max_tokens=1000
    )
    
    return respuesta_final.choices[0].message.content


# Test 1: Búsqueda web
print("="*60)
print("TEST 1 — Búsqueda web")
respuesta1 = ejecutar_con_function_calling(
    "¿Cuáles son las novedades más recientes de Microsoft Fabric en 2024?"
)
print(f"\n✅ RESPUESTA FINAL:\n{respuesta1}")

TEST 1 — Búsqueda web

>>> PROMPT: ¿Cuáles son las novedades más recientes de Microsoft Fabric en 2024?

🔧 Función llamada: buscar_web
   Argumentos: {'query': 'Microsoft Fabric novedades 2024', 'num_resultados': 3}
   Resultado: {'nota': 'Modo demo - configura BING_SEARCH_KEY para búsqueda real', 'query': 'Microsoft Fabric novedades 2024', 'resultados': [{'titulo': 'Resultado demo para: Microsoft Fabric novedades 2024', 'snip...

✅ RESPUESTA FINAL:
Actualmente, estoy en modo de demostración y no tengo acceso a contenido en tiempo real. Sin embargo, puedes consultar este [enlace de ejemplo](https://example.com) para explorar sobre las novedades de Microsoft Fabric en 2024. También, si tienes preguntas específicas, estaré encantado de ayudarte con la información general sobre Microsoft Fabric.


In [24]:
# Test 2: Función custom de análisis de pipeline (⭐)
print("="*60)
print("TEST 2 — Función custom: análisis de pipeline")
respuesta2 = ejecutar_con_function_calling(
    "Tengo un pipeline en Azure Data Factory que procesa 250 GB de datos diariamente. ¿Cuánto me va a costar al mes y qué optimizaciones me recomiendas?"
)
print(f"\n✅ RESPUESTA FINAL:\n{respuesta2}")

TEST 2 — Función custom: análisis de pipeline

>>> PROMPT: Tengo un pipeline en Azure Data Factory que procesa 250 GB de datos diariamente. ¿Cuánto me va a costar al mes y qué optimizaciones me recomiendas?

🔧 Función llamada: analizar_pipeline_datos
   Argumentos: {'volumen_gb': 250, 'frecuencia': 'daily', 'tecnologia': 'adf'}
   Resultado: {'analisis': {'volumen_gb': 250, 'frecuencia': 'daily', 'tecnologia': 'adf', 'coste_estimado_mensual_usd': 900.0, 'tiempo_ejecucion_minutos': 300.0, 'ejecuciones_mensuales': 30, 'recomendaciones': ['C...

✅ RESPUESTA FINAL:
Procesar 250 GB de datos diariamente en Azure Data Factory podría tener un coste estimado de **900 USD al mes**, suponiendo una ejecución diaria durante 30 días (300 minutos totales).

### Recomendaciones de optimización:
1. **Particionado de datos**: Divide los datos en particiones más pequeñas para optimizar el rendimiento y reducir el tiempo de procesamiento.
2. **Uso de Azure Synapse o Azure Databricks (Spark)**: Comparte la

In [25]:
# Test 3: Combinación de búsqueda web + función custom (⭐⭐)
print("="*60)
print("TEST 3 — Combinación: búsqueda web + función custom")
respuesta3 = ejecutar_con_function_calling(
    "Estoy migrando un pipeline de Spark que procesa 500 GB semanalmente a Microsoft Fabric. ¿Cuánto ahorraría y qué dice la documentación reciente sobre la migración?"
)
print(f"\n✅ RESPUESTA FINAL:\n{respuesta3}")

TEST 3 — Combinación: búsqueda web + función custom

>>> PROMPT: Estoy migrando un pipeline de Spark que procesa 500 GB semanalmente a Microsoft Fabric. ¿Cuánto ahorraría y qué dice la documentación reciente sobre la migración?

🔧 Función llamada: analizar_pipeline_datos
   Argumentos: {'volumen_gb': 500, 'frecuencia': 'weekly', 'tecnologia': 'fabric'}
   Resultado: {'analisis': {'volumen_gb': 500, 'frecuencia': 'weekly', 'tecnologia': 'fabric', 'coste_estimado_mensual_usd': 120.0, 'tiempo_ejecucion_minutos': 200.0, 'ejecuciones_mensuales': 4, 'recomendaciones': ...

🔧 Función llamada: buscar_web
   Argumentos: {'query': 'migración de pipelines de Spark a Microsoft Fabric', 'num_resultados': 3}
   Resultado: {'nota': 'Modo demo - configura BING_SEARCH_KEY para búsqueda real', 'query': 'migración de pipelines de Spark a Microsoft Fabric', 'resultados': [{'titulo': 'Resultado demo para: migración de pipelin...

✅ RESPUESTA FINAL:
En la migración de un pipeline que procesa 500 GB semanalm

---
## Conclusiones y Problemas Encontrados

### Conclusiones
1. **Niveles de razonamiento**: El nivel `high` produce respuestas más exhaustivas y estructuradas en problemas de optimización técnica compleja, pero con un coste de tokens 3-4x mayor que `low`. Para preguntas simples, `low` es suficiente.
2. **Function Calling**: El modelo identifica correctamente cuándo necesita herramientas externas sin instrucción explícita. El ciclo tool_call → ejecución → respuesta final funciona de forma transparente.
3. **Función custom**: Permite encapsular lógica de negocio compleja y exponerla al modelo como una herramienta, combinando razonamiento del LLM con cálculos deterministas.

### Problemas Encontrados
- **API de razonamiento**: Los modelos o1/o3-mini no aceptan `temperature` ni `system` messages directamente. Se deben pasar las instrucciones en el `user` message.
- **Latencia en high reasoning**: El nivel `high` puede tardar 20-30 segundos. Para producción considerar streaming o respuestas asíncronas.
- **Bing Search sin API key**: En entornos de desarrollo sin la key de Bing, el mock devuelve datos de ejemplo. La arquitectura es idéntica con la key real.